# MindeesAI — Kaggle GPU training notebook

Trains the `home-11gb` variant (~280M params) on a Kaggle T4 / P100 GPU. One 12-hour session ≈ 150-300k steps, which is **more than a month** of the GH Actions CPU cron.

## Before you run

1. **Right sidebar → Notebook options:**
   - **Accelerator: GPU T4 ×2** (or P100, both work)
   - **Internet: ON** (needed for HF + dataset downloads)
   - **Persistence: Files only** (saves your /kaggle/working between sessions)
2. **Add-ons → Secrets → Add new secret:**
   - **Label**: `HF_TOKEN`
   - **Value**: your HuggingFace **write-scoped** token from https://huggingface.co/settings/tokens

## What this notebook does

1. Clone `aashir-athar/mindeesai` repo
2. Pull `HF_TOKEN` from Kaggle Secrets
3. Resume-fetch prior `base.bin` + `torch-resume.pt` + `tokenizer.json` from HF revision `kaggle-weekly`
4. Train `home-11gb` for 200,000 steps OR until the 12-hour cap, whichever comes first
5. Push the new checkpoint to HF revision `kaggle-weekly` (isolated branch — does **NOT** touch `main` or `small-weekly`)

**Branch safety:** This notebook only writes to the `kaggle-weekly` revision. Your local-trained `main` checkpoint and the GH Actions `small-weekly` checkpoint are untouched.

## Cell 1 — Clone repo + install dependencies

Takes ~2-3 min. Pulls the latest `main` from GitHub so it always has your most recent recipe + variant definitions.

In [1]:
!rm -rf /kaggle/working/mindeesai
!git clone -q https://github.com/aashir-athar/mindeesai.git /kaggle/working/mindeesai
%cd /kaggle/working/mindeesai
!pip install -q -r scripts/train/requirements.txt
!pip install -q huggingface_hub

import torch
print(f"PyTorch {torch.__version__} · CUDA available: {torch.cuda.is_available()} · GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

/kaggle/working/mindeesai
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.1/763.1 kB 19.4 MB/s eta 0:00:00
PyTorch 2.10.0+cu128 · CUDA available: True · GPU: Tesla T4


## Cell 2 — Load `HF_TOKEN` from Kaggle Secrets

In [2]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]

assert os.environ["HF_TOKEN"].startswith("hf_"), "HF_TOKEN does not look like a valid HuggingFace token — check Add-ons → Secrets"
print(f"HF_TOKEN loaded (length {len(os.environ['HF_TOKEN'])} chars)")

HF_TOKEN loaded (length 37 chars)


## Cell 3 — Resume from HF `kaggle-weekly` revision

On the first run this 404s on every file (nothing pushed yet) and we train from random init. From the second run onwards it picks up the saved AdamW moments + LR schedule + step count and continues.

If you want to start from your local-trained `main` checkpoint instead (e.g. continue training a home-11gb that you already trained on your RTX), change `REVISION` below to `"main"`.

In [3]:
import os
from huggingface_hub import hf_hub_download

REPO = "aashir-athar/mindeesai-base"
REVISION = "kaggle-weekly"   # change to "main" if you want to resume the local-trained checkpoint

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("tokenizer", exist_ok=True)

for filename, local_dir in [
    ("base.bin",        "checkpoints"),
    ("torch-resume.pt", "checkpoints"),
    ("tokenizer.json",  "tokenizer"),
]:
    try:
        p = hf_hub_download(repo_id=REPO, filename=filename, revision=REVISION,
                            local_dir=local_dir, local_dir_use_symlinks=False,
                            token=os.environ["HF_TOKEN"])
        print(f"  ✓ {filename}: {os.path.getsize(p):,} bytes → {p}")
    except Exception as e:
        msg = str(e).lower()
        if "404" in msg or "not found" in msg or "entry not found" in msg:
            print(f"  · {filename}: not on HF yet (first run, will train fresh)")
        else:
            print(f"  ✗ {filename}: {e}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


  · base.bin: not on HF yet (first run, will train fresh)
  · torch-resume.pt: not on HF yet (first run, will train fresh)
  · tokenizer.json: not on HF yet (first run, will train fresh)


## Cell 4 — Train

Default: `home-11gb` variant (~280M params), batch 4 × grad-accum 2 = effective batch 8, fp16 + gradient checkpointing, full `mix-broadbrain.json` dataset recipe (21 datasets covering code / math / reasoning / chat / sales / marketing / psychology / business / UI-UX / web3 / RN).

Step time on T4 / P100 at this size is roughly **0.5-1.0 sec/step** depending on data source and grad-ckpt overhead. 200,000 steps in 12h is the target ceiling; expect to actually finish 100,000-200,000 depending on GPU assignment.

If you see CUDA OOM, drop `--batch` to 2 (or 1) and bump `--grad-accum` to keep the effective batch.

In [4]:
!python scripts/train/pretrain.py \
    --variant home-11gb \
    --corpus scripts/data/corpus.txt \
    --tokenizer tokenizer/tokenizer.json \
    --steps 200000 \
    --batch 4 \
    --grad-accum 2 \
    --lr 3e-4 \
    --warmup 1000 \
    --wd 0.1 \
    --val-frac 0.05 \
    --val-every 500 \
    --ckpt-every 1000 \
    --base-weight 1.0 \
    --distill-weight 4.0 \
    --completion-only-loss 1 \
    --persona-loss-weight 0.05 \
    --mix-config scripts/data/mix-broadbrain.json \
    --resume checkpoints/torch-resume.pt \
    --amp \
    --grad-ckpt \
    --log data/training-metrics.jsonl \
    --out checkpoints/base.bin \
    --torch-ckpt checkpoints/torch-resume.pt

device=cuda  variant=home-11gb  use_moe=False  use_mla=True  use_mtp=True
tokenizing corpus: scripts/data/corpus.txt
Traceback (most recent call last):
  File "/kaggle/working/mindeesai/scripts/train/pretrain.py", line 1258, in <module>
    main()
  File "/kaggle/working/mindeesai/scripts/train/pretrain.py", line 991, in main
    base_tokens = load_tokens(args.corpus, args.tokenizer, device="cpu")
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/mindeesai/scripts/train/pretrain.py", line 546, in load_tokens
    _vocab, by_pair = load_bpe(tokenizer_path)
                      ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/mindeesai/scripts/train/pretrain.py", line 405, in load_bpe
    raw = json.loads(Path(tokenizer_path).read_text(encoding="utf-8"))
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/pathlib.py", line 1027, in read_text
    with self.open(mode='r', encoding=encoding, errors=

## Cell 5 — Push trained checkpoint back to HF `kaggle-weekly`

**Safety**: this only writes to the `kaggle-weekly` revision. Your `main` (local RTX work) and `small-weekly` (GH Actions cron) revisions stay untouched.

If you want this Kaggle run to become your production checkpoint, set `HF_MODEL_REVISION=kaggle-weekly` on Vercel (NOT here).

In [5]:
import os
from huggingface_hub import HfApi, create_branch

REPO = "aashir-athar/mindeesai-base"
TARGET_REVISION = "kaggle-weekly"   # ⚠️ NEVER change to "main" by accident

api = HfApi(token=os.environ["HF_TOKEN"])

# Make sure the branch exists (idempotent)
try:
    create_branch(REPO, branch=TARGET_REVISION, exist_ok=True, token=os.environ["HF_TOKEN"])
except Exception as e:
    print(f"create_branch note: {e}")

uploaded = 0
for local, in_repo in [
    ("checkpoints/base.bin",         "base.bin"),
    ("checkpoints/torch-resume.pt",  "torch-resume.pt"),
    ("tokenizer/tokenizer.json",     "tokenizer.json"),
    ("data/training-metrics.jsonl",  "training-metrics.jsonl"),
]:
    if not os.path.exists(local):
        print(f"  · {local}: not found, skipping")
        continue
    try:
        api.upload_file(
            path_or_fileobj=local,
            path_in_repo=in_repo,
            repo_id=REPO,
            revision=TARGET_REVISION,
            commit_message=f"Kaggle session — {in_repo}",
        )
        print(f"  ✓ {local} → {REPO}@{TARGET_REVISION}:{in_repo}")
        uploaded += 1
    except Exception as e:
        print(f"  ✗ {local}: {e}")

print(f"\nUploaded {uploaded} files. View on HF: https://huggingface.co/{REPO}/tree/{TARGET_REVISION}")
print("To use this in production: set HF_MODEL_REVISION=kaggle-weekly on Vercel.")

  · checkpoints/base.bin: not found, skipping
  · checkpoints/torch-resume.pt: not found, skipping
  · tokenizer/tokenizer.json: not found, skipping
  · data/training-metrics.jsonl: not found, skipping

Uploaded 0 files. View on HF: https://huggingface.co/aashir-athar/mindeesai-base/tree/kaggle-weekly
To use this in production: set HF_MODEL_REVISION=kaggle-weekly on Vercel.


## Done

Hit **Save Version** (top right) before closing the tab so your output gets preserved in Kaggle's history. The HF push above already preserved the trained checkpoint — you can close this and walk away.

Next session: just run this notebook again. Cell 3 will fetch the checkpoint you just pushed and Cell 4 will resume from where you left off.